# 02 - Base LM (Vaswani 2017) - emotion paraphraser

Trains the decoder-only Vaswani transformer LM on the **TRAIN** captions, conditioned on the emotion tag. Prefers your `modules/vaswani_lm_module.py`; falls back to an inline equivalent so the notebook is runnable. Checkpoints every epoch and **auto-resumes** from the latest.

**Chain in:** `artifacts/train.csv`. **Chain out:** `checkpoints/vaswani_lm/vaswani_lm_epoch10.pt`.

In [12]:

# Mount Google Drive
!fusermount -u /content/drive 2>/dev/null
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import os, glob, json, math, textwrap, random, re
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE=='cuda' else torch.float32
ROOT ='/content/drive/MyDrive/Text2ImageNarration'
DATA = f'{ROOT}/data'; PROC = f'{ROOT}/data/train'
ART  = f'{ROOT}/artifacts'; CKPT = f'{ROOT}/checkpoints'
GEN  = f'{ROOT}/outputs/generated'; BASE = f'{ROOT}/baselines/xlxmert'; EVALD = f'{ROOT}/outputs/eval'
for d in (DATA, ART, CKPT, GEN, EVALD): os.makedirs(d, exist_ok=True)
SEED = 42; N_TEST = 200
def img_dir():
    return PROC if (os.path.isdir(PROC) and glob.glob(f'{PROC}/*')) else f'{DATA}/anime_images'
from PIL import Image, ImageDraw, ImageFont
print('device', DEVICE, '| ROOT', ROOT, '| images', img_dir())

device cuda | ROOT /content/drive/MyDrive/Text2ImageNarration | images /content/drive/MyDrive/Text2ImageNarration/data/train


In [15]:
# Prefer the Drive module; otherwise define an inline decoder-only LM
import torch.nn as nn
USE_MODULE = False
try:
    import sys; sys.path.insert(0, f'{ROOT}/modules')
    from vaswani_lm_module import VaswaniTransformerLM, CaptionDataset, paraphrase
    USE_MODULE = True; print('Using modules/vaswani_lm_module.py')
except Exception as e:
    print('Inline Vaswani LM (module not found):', e)
    class CaptionDataset(torch.utils.data.Dataset):
        def __init__(self, df, max_len=64, min_freq=1):
            self.max_len = max_len
            from collections import Counter
            toks = [self._tok(c) for c in df['caption'].astype(str)]
            cnt = Counter(t for ts in toks for t in ts)
            self.itos = ['<pad>','<bos>','<eos>','<unk>'] + [w for w,c in cnt.items() if c >= min_freq]
            self.stoi = {w:i for i,w in enumerate(self.itos)}
            ecol = df['emotion'].astype(str).str.lower() if 'emotion' in df else None
            emos = sorted(ecol.unique()) if ecol is not None else ['neutral']
            self.emo2id = {e:i for i,e in enumerate(emos)}
            self.examples = []
            for j,(c) in enumerate(df['caption'].astype(str)):
                e = ecol.iloc[j] if ecol is not None else 'neutral'
                ids = [1] + [self.stoi.get(t,3) for t in self._tok(c)][:max_len-2] + [2]
                ids = ids + [0]*(max_len-len(ids))
                self.examples.append((ids, self.emo2id.get(str(e).lower(),0)))
        @staticmethod
        def _tok(s): return re.findall(r"[a-z0-9']+", str(s).lower())
        @property
        def vocab_size(self): return len(self.itos)
        def vocab_dict(self): return {'itos':self.itos,'emo2id':self.emo2id}
        def __len__(self): return len(self.examples)
        def __getitem__(self, i):
            ids,e = self.examples[i]; t = torch.tensor(ids)
            return t[:-1], t[1:], torch.tensor(e)
    class VaswaniTransformerLM(nn.Module):
        def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=6, d_ff=1024,
                     max_len=64, dropout=0.1, n_emotions=32):
            super().__init__()
            self.tok = nn.Embedding(vocab_size, d_model, padding_idx=0)
            self.pos = nn.Embedding(max_len, d_model)
            self.emo = nn.Embedding(n_emotions, d_model)
            layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True)
            self.enc = nn.TransformerEncoder(layer, n_layers)
            self.ln = nn.LayerNorm(d_model); self.head = nn.Linear(d_model, vocab_size)
            self.max_len = max_len
        def forward(self, x, emotion=None):
            B,L = x.shape
            h = self.tok(x) + self.pos(torch.arange(L, device=x.device))[None]
            if emotion is not None: h = h + self.emo(emotion)[:,None]
            mask = torch.triu(torch.full((L,L), float('-inf'), device=x.device), diagonal=1)
            h = self.enc(h, mask=mask, src_key_padding_mask=(x==0))
            return self.head(self.ln(h))
    @torch.no_grad()
    def paraphrase(model, caption, emotion, k=2, max_new=40, temp=0.9):
        model.eval(); outs=[]
        emo = getattr(model,'emo2id',{}).get(str(emotion).lower(),0)
        for _ in range(k):
            ids = [1] + [model.stoi.get(t,3) for t in CaptionDataset._tok(caption)][:20]
            for _ in range(max_new):
                x = torch.tensor(ids[-model.max_len:], device=DEVICE)[None]
                logits = model(x, emotion=torch.tensor([emo], device=DEVICE))[0,-1] / temp
                nxt = int(torch.multinomial(torch.softmax(logits,-1), 1))
                if nxt == 2: break
                ids.append(nxt)
            outs.append(' '.join(model.itos[i] for i in ids if i > 3))
        return outs
print('vaswani primitives ready (module=%s)' % USE_MODULE)

Inline Vaswani LM (module not found): No module named 'vaswani_lm_module'
vaswani primitives ready (module=False)


In [16]:
# Build dataset from TRAIN split + train with per-epoch checkpoint / auto-resume
import pandas as pd
df = pd.read_csv(f'{ART}/train.csv')
ds = CaptionDataset(df, max_len=64)
cfg = dict(vocab_size=ds.vocab_size, d_model=256, n_heads=8, n_layers=6, d_ff=1024,
           max_len=64, dropout=0.1, n_emotions=max(len(ds.emo2id), 32))
model = VaswaniTransformerLM(**cfg).to(DEVICE)
if not USE_MODULE:
    model.itos, model.stoi, model.emo2id = ds.itos, ds.stoi, ds.emo2id
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M params | vocab {ds.vocab_size} | emotions {len(ds.emo2id)}')

from torch.utils.data import DataLoader
VDIR = f'{CKPT}/vaswani_lm'; os.makedirs(VDIR, exist_ok=True)
EPOCHS = 10
cks = sorted(glob.glob(f'{VDIR}/vaswani_lm_epoch*.pt'),
             key=lambda p: int(re.findall(r'epoch(\d+)', p)[0]) if re.findall(r'epoch(\d+)', p) else 0)
start = 0
if cks:
    st = torch.load(cks[-1], map_location=DEVICE)
    try: model.load_state_dict(st['weights']); start = int(st.get('epoch', 0)); print('resumed from', cks[-1])
    except Exception as e: print('resume failed, training fresh:', e)
loader = DataLoader(ds, batch_size=64, shuffle=True, drop_last=True)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
for epoch in range(start, EPOCHS):
    model.train(); tot=n=0
    for x,y,e in loader:
        x,y,e = x.to(DEVICE), y.to(DEVICE), e.to(DEVICE)
        logits = model(x, emotion=e)
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=0)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); n += 1
    torch.save({'cfg':cfg, 'weights':model.state_dict(), 'vocab':ds.vocab_dict(), 'epoch':epoch+1},
               f'{VDIR}/vaswani_lm_epoch{epoch+1}.pt')
    print(f'epoch {epoch+1:2d}  loss {tot/max(n,1):.4f}  (checkpoint saved)')
print('done ->', f'{VDIR}/vaswani_lm_epoch{EPOCHS}.pt')

4.79M params | vocab 57 | emotions 2
resumed from /content/drive/MyDrive/Text2ImageNarration/checkpoints/vaswani_lm/vaswani_lm_epoch10.pt
done -> /content/drive/MyDrive/Text2ImageNarration/checkpoints/vaswani_lm/vaswani_lm_epoch10.pt


In [17]:
# Validation: held-out perplexity + sample emotion paraphrases
import math
from torch.utils.data import DataLoader
model.eval(); tot=n=0
with torch.no_grad():
    for x,y,e in DataLoader(ds, batch_size=64):
        x,y,e = x.to(DEVICE), y.to(DEVICE), e.to(DEVICE)
        logits = model(x, emotion=e)
        tot += torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=0).item(); n += 1
print(f'perplexity: {math.exp(tot/max(n,1)):.2f}')
import pandas as pd
for _,r in pd.read_csv(f'{ART}/train.csv').sample(min(3,len(df)), random_state=0).iterrows():
    print('-'*50, chr(10), 'EMO', r.get('emotion','?'), '| ORIG', r['caption'])
    try:
        for k,p in enumerate(paraphrase(model, r['caption'], r.get('emotion','calm'), k=2), 1): print(f'  PARA-{k}:', p)
    except Exception as ex: print('  paraphrase err:', ex)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


perplexity: 72.27
-------------------------------------------------- 
 EMO fear | ORIG a cartoon character with a hat and glasses 
  PARA-1: a cartoon character with a hat and glasses it's of glasses bikini head girl's of that tattoo pink character red pink glasses red dress bikini of it's is mask a dress doll close painting painted it tattoo wearing with red painted close painted character girl's up
  PARA-2: a cartoon character with a hat and glasses red pink glasses is character mouth bikini glasses character close girl's blue girl's cat painted of painted red red glasses toothbrush painted and of cat cat white face red bikini bikini red painted holding painted girl's character
-------------------------------------------------- 
 EMO fear | ORIG a woman in a black and white photo 
  PARA-1: a woman in a black and white photo cartoon blue it's pink dress their mask holding bikini sunglasses person phone sunglasses on mask painted front on with of character blue girl's red flower mout

## TextStreamer streaming generation  (proposal step a)
Demonstrates token-by-token autoregressive generation streamed through HuggingFace `TextStreamer` (via a small adapter over the Vaswani vocabulary). The Vaswani LM itself uses a custom word-level tokenizer; `AutoTokenizer` is used by the Mistral TaleCrafter backend in notebook 04.

In [18]:
# TextStreamer streaming-generation demo over the trained Vaswani LM
itos = getattr(model, 'itos', None)
if itos is None:
    print('TextStreamer demo needs the inline-model vocab (model.itos); skipped for the Drive-module model.')
else:
    try:
        from transformers import TextStreamer
        class _VocabShim:           # minimal tokenizer adapter so TextStreamer can decode our ids
            def __init__(self, itos): self.itos = itos
            def decode(self, ids, **kw):
                if hasattr(ids, 'tolist'): ids = ids.tolist()
                if isinstance(ids, int): ids = [ids]
                return ' '.join(self.itos[i] for i in ids if i > 3) + ' '
        streamer = TextStreamer(_VocabShim(itos), skip_prompt=False)
        emo = model.emo2id.get('happy', 0)
        print('streaming a "happy" generation:')
        ids = [1]                    # <bos>
        for _ in range(30):
            x = torch.tensor(ids[-model.max_len:], device=DEVICE)[None]
            logits = model(x, emotion=torch.tensor([emo], device=DEVICE))[0, -1] / 0.9
            nxt = int(torch.multinomial(torch.softmax(logits, -1), 1))
            streamer.put(torch.tensor([[nxt]]))
            ids.append(nxt)
            if nxt == 2: break        # <eos>
        streamer.end()
    except Exception as e:
        print('TextStreamer unavailable, manual stream instead:', e)
        emo = model.emo2id.get('happy', 0); ids = [1]
        for _ in range(30):
            x = torch.tensor(ids[-model.max_len:], device=DEVICE)[None]
            logits = model(x, emotion=torch.tensor([emo], device=DEVICE))[0, -1] / 0.9
            nxt = int(torch.multinomial(torch.softmax(logits, -1), 1))
            if nxt == 2: break
            if nxt > 3: print(model.itos[nxt], end=' ', flush=True)
            ids.append(nxt)
        print()

streaming a "happy" generation:
cat animal cat red of glasses mask girl's camera photo her hair doll flower doll hair girl red picture person character photo cell tattoo dress mask cell face tattoo 


**Next:** `03_StyleBoost_FineTune`.